# Compact Seasonal Variation Visualizations

Based on the requirement for a compact visualization (maximum of two figures fitting on an A4 page), this notebook provides two highly informative, composite figures to summarize the seasonal variation.

1. **Figure 1: Spatial Pattern of Seasons** - A 2x2 grid of maps showing the spatial distribution of DO for each of the four seasons in a representative year.
2. **Figure 2: Distribution and Trends** - A composite figure containing a boxplot for overall seasonal variation and a time series of the long-term seasonal means.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata
from pathlib import Path
import sys
import seaborn as sns

# Ensure project paths are available
project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils.paths import ProjectPaths

def load_seasonal_data(season, layer, target_year):
    subset_dir = ProjectPaths.get_subset_dir(season, layer)
    yearly_do_file = subset_dir / "yearly_do_df.csv"
    
    if not yearly_do_file.exists():
        return None
        
    yearly_do_df = pd.read_csv(yearly_do_file, dtype={'zettaicode': str})
    yearly_do_df.set_index('zettaicode', inplace=True)
    
    if str(target_year) in yearly_do_df.columns:
        target_year = str(target_year)
    elif int(target_year) in yearly_do_df.columns:
        target_year = int(target_year)
    else:
        return None
        
    do_values = yearly_do_df[[target_year]].dropna().rename(columns={target_year: 'do_'})
    
    merged_gdf = gpd.read_feather(ProjectPaths.MERGED_DATA_FILE)
    stations_gdf = merged_gdf[['zettaicode', 'longitude', 'latitude']].drop_duplicates().set_index('zettaicode')
    
    return do_values.join(stations_gdf, how='inner')

def get_yearly_means_for_season(season, layer):
    subset_dir = ProjectPaths.get_subset_dir(season, layer)
    yearly_do_file = subset_dir / "yearly_do_df.csv"
    if not yearly_do_file.exists():
        return pd.Series(dtype=float)
        
    yearly_do_df = pd.read_csv(yearly_do_file, dtype={'zettaicode': str})
    yearly_do_df.set_index('zettaicode', inplace=True)
    
    means = yearly_do_df.mean(axis=0)
    means.index = pd.to_numeric(means.index, errors='coerce')
    means = means.dropna().sort_index()
    return means

## Figure 1: Spatial Distribution Across Four Seasons

In [ ]:
# Configuration
layer = 'bottom'
target_year = '1991'
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']

fig, axes = plt.subplots(2, 2, figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(f'Spatial Distribution of DO in {target_year} ({layer} layer)', fontsize=16)
axes = axes.flatten()

extent = [130.6, 135.6, 32.75, 35.1]
grid_x, grid_y = np.mgrid[extent[0]:extent[1]:200j, extent[2]:extent[3]:200j]

cm = None
for i, season in enumerate(seasons):
    ax = axes[i]
    ax.set_extent(extent)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.LAND, color='lightgray')
    
    df = load_seasonal_data(season, layer, target_year)
    if df is not None and len(df) > 3:
        points = df[['longitude', 'latitude']].values
        values = df['do_'].values
        grid_z = griddata(points, values, (grid_x, grid_y), method='linear')
        
        # Use a consistent vmin/vmax for comparison across subplots if possible
        cm = ax.contourf(grid_x, grid_y, grid_z, levels=15, cmap='viridis', extend='both', vmin=1, vmax=9)
        ax.scatter(points[:, 0], points[:, 1], c='black', s=2, alpha=0.3)
        
    ax.set_title(season, fontsize=12)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
if cm:
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
    fig.colorbar(cm, cax=cbar_ax, orientation='horizontal', label='Dissolved Oxygen (DO)')
plt.show()

## Figure 2: Seasonal Distributions and Long-Term Trends

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Spring': 'green', 'Summer': 'red', 'Autumn': 'orange', 'Winter': 'blue'}

# Subplot 1: Boxplot of distributions across a wide range of years (or just target year)
# We will pull target_year data for simplicity in this example
all_seasons_data = []
for s in seasons:
    df = load_seasonal_data(s, layer, target_year)
    if df is not None:
        df['Season'] = s
        all_seasons_data.append(df)

if all_seasons_data:
    combined_df = pd.concat(all_seasons_data, ignore_index=True)
    sns.boxplot(x='Season', y='do_', data=combined_df, order=seasons, palette=colors, ax=ax1, width=0.5)
    sns.stripplot(x='Season', y='do_', data=combined_df, order=seasons, color='black', alpha=0.2, jitter=True, ax=ax1)
    ax1.set_title(f'DO Distribution by Season ({target_year})')
    ax1.set_ylabel('Dissolved Oxygen (DO)')
    ax1.set_xlabel('')

# Subplot 2: Long-term trends of basin seasonal averages
for s in seasons:
    means = get_yearly_means_for_season(s, layer)
    if not means.empty:
        ax2.plot(means.index, means.values, marker='o', markersize=4, label=s, color=colors[s], alpha=0.8)

ax2.set_title('Long-Term Trend of Seasonal Mean DO')
ax2.set_xlabel('Year')
ax2.set_ylabel('Mean Dissolved Oxygen')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(title='Season', loc='lower left')

plt.tight_layout()
plt.show()